In [ ]:
import pynq import Overlay, allocate
import numpy as np
import time

In [ ]:
# Software convolution for verification
def conv2d_sw(image, kernel, norm_shift):
    H, W = image.shape
    K = kernel.shape[0]

    out_h = H - K + 1
    out_w = W - K + 1

    out = np.zeros((out_h, out_w), dtype=np.uint8)

    for i in range(out_h):
        for j in range(out_w):
            window = image[i:i+K, j:j+K]
            s = int(np.sum(window * kernel))

            # match HLS
            bias = (1 << (norm_shift - 1)) if norm_shift > 0 else 0
            result = (s + bias) >> norm_shift

            # CLAMP
            result = max(0, min(255, result))

            out[i, j] = result
    
    return out

In [ ]:
# Initialize overlay and IPs
overlay = Overlay("../vivado/fpga_2d_convolution.bit")
dma = overlay.axi_dma
hls_ip = overlay.conv_dataflow_stream

In [ ]:
# Helper functions to pack/unpack 8-bit pixels into 32-bit words for DMA transfer
def pack_pixels(image):
    flat = image.flatten()
    packed = np.zeros(( (len(flat)+3)//4, ), dtype=np.uint32)

    word = 0
    count = 0
    idx = 0

    for px in flat:
        word |= int(px) << (8 * count)
        count += 1
        
        if count == 4:
            packed[idx] = word
            idx += 1
            word = 0
            count = 0

    if count != 0:
        packed[idx] = word

    return packed

def unpack_pixels(packed, total_pixels):
    out = np.zeros(total_pixels, dtype=np.uint8)
    idx = 0

    finished = False
    for word in packed:
        for i in range(4):
            if idx < total_pixels:
                out[idx] = (word >> (8 * i)) & 0xFF
                idx += 1
            
            if idx >= total_pixels:
                finished = True
                break
            
        if finished:
            break

    return out

In [ ]:
# Main function to run convolution on hardware
def conv2d_hw(image, kernel, norm_shift):
    flat_kernel = kernel.flatten()

    # 1. Write kernel
    for i in range(len(flat_kernel)):
        hls_ip.write(0x10 + 0x8*i, int(flat_kernel[i]))

    # 2. Write params
    H, W = image.shape
    hls_ip.write(0x58, H) # write height to Height register
    hls_ip.write(0x60, W) # write width to Width register
    hls_ip.write(0x68, norm_shift) # write norm shift to NormShift register

    # ===== TIMING START =====
    t0 = time.time()

    # 3. Pack input
    t_pack_start = time.time()
    packed_input = pack_pixels(image)
    t_pack_end = time.time()

    # 4. Allocate output buffer
    out_h = H - 2
    out_w = W - 2
    total_pixels = out_h * out_w
    packed_out_size = (total_pixels + 3) // 4
    output_buffer = np.zeros(packed_out_size, dtype=np.uint32)

    # 5. Start IP
    hls_ip.write(0x00, 1)

    # 6. DMA + compute
    t_hw_start = time.time()

    dma.sendchannel.transfer(packed_input)
    dma.recvchannel.transfer(output_buffer)

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    t_hw_end = time.time()

    # 7. Unpack output
    t_unpack_start = time.time()
    output = unpack_pixels(output_buffer, total_pixels)
    t_unpack_end = time.time()

    t1 = time.time()

    return {
        "output": output.reshape(out_h, out_w),
        "total_time": t1 - t0,
        "hw_time": t_hw_end - t_hw_start,
        "pack_time": t_pack_end - t_pack_start,
        "unpack_time": t_unpack_end - t_unpack_start
    }

In [ ]:
# Kernels
import numpy as np

# Sobel X, Sobel Y, and Gaussian kernels
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.int32)

sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.int32)

gaussian = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.int32)

In [ ]:
# Benchmarking
import time

sizes = [2**i for i in range(2, 10)]  # 4 -> 512

def benchmark(kernel, norm_shift, label):
    sw_times = []
    hw_times = []

    for size in sizes:
        image = np.random.randint(0, 256, (size, size), dtype=np.uint8)

        # --- Software ---
        t0 = time.time()
        out_sw = conv2d_sw(image, kernel, norm_shift)
        t1 = time.time()

        # --- Hardware ---
        result = conv2d_hw(image, kernel, norm_shift)
        out_hw = result["output"]

        # correctness check
        assert np.array_equal(out_sw, out_hw), f"Mismatch at size {size}"

        sw_times.append(t1 - t0)
        hw_times.append(result["total_time"])

        print(f"{label} | {size}x{size} → OK")

    return sw_times, hw_times

In [ ]:
# Run all Kernels
results = {}

results["Sobel X"] = benchmark(sobel_x, norm_shift=0, label="Sobel X")
results["Sobel Y"] = benchmark(sobel_y, norm_shift=0, label="Sobel Y")
results["Gaussian"] = benchmark(gaussian, norm_shift=4, label="Gaussian")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for name, (sw_times, hw_times) in results.items():
    plt.plot(sizes, sw_times, '--', label=f"{name} SW")
    plt.plot(sizes, hw_times, '-', label=f"{name} HW")

plt.xlabel("Image Size")
plt.ylabel("Time (seconds)")
plt.title("2D Convolution Performance Scaling")
plt.legend()
plt.grid()
plt.xscale("log", base=2)

plt.show()

In [ ]:
# Plot speedup
speedup = [sw/hw for sw, hw in zip(sw_times, hw_times)]
plt.plot(sizes, speedup)
plt.title("Speedup (SW / HW)")

The hardware implementation includes data packing/unpacking and DMA transfers. 
For smaller inputs, overhead dominates performance. 
As input size increases, the streaming architecture achieves higher efficiency 
due to its pipelined II=1 design.